In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle

In [21]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [22]:
# preprocess the data 
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

In [23]:
# Encode categorical variables
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [24]:
# One-hot encode 'Geography'

# 1. Initialize the encoder. 
# 'handle_unknown=ignore' use kiya hai taaki future mein naya desh aane par app crash na ho.
onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')

# 2. Data ko encode karke array mein badalna.
# .toarray() lagaya hai taaki sparse (compressed) matrix khul kar normal 2D array ban jaye. aur is .toarray() ki wajah se sparse_output=false
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()

# 3. Array ko wapas Pandas table (DataFrame) mein convert karna.
# get_feature_names_out() apne aap 'Geography_France', 'Geography_Spain' jaise naam de dega.
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [25]:
# combine one-hot encoded columns with original data 
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [26]:
# Split the data into features and target

X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']
y

0       101348.88
1       112542.58
2       113931.57
3        93826.63
4        79084.10
          ...    
9995     96270.64
9996    101699.77
9997     42085.58
9998     92888.52
9999     38190.78
Name: EstimatedSalary, Length: 10000, dtype: float64

In [27]:
# split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [28]:
## scale these features 

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# save the encoder and scaler for future use
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(onehot_encoder_geo, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

## ANN Regression problem statement

In [30]:
import tensorflow as tf 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [ ]:
# Build the model 
model =  Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1)]  # output layer for regression and default activation='linear'
)

# compile the model 
model.compile(optimizer='adam', loss='mean_absolute_error', metrics=['mae']) # mae =  MeanAbsoluteError
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 64)                832       
                                                                 
 dense_7 (Dense)             (None, 32)                2080      
                                                                 
 dense_8 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [32]:
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

#setup tensorboard

log_dir = "regressionlogs/fits/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


In [33]:
# setup early stopping

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [34]:
# train the model 
history = model.fit(X_train, y_train, validation_data=(X_test,y_test), epochs=100, callbacks=[early_stopping_callback, tensorboard_callback])

Epoch 1/100


250/250 [==============================] - 3s 6ms/step - loss: 70849.9297 - mae: 70849.9297 - val_loss: 67030.9219 - val_mae: 67030.9219
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 65101.7617 - mae: 65101.7617 - val_loss: 60074.2031 - val_mae: 60074.2031
Epoch 3/100
250/250 [==============================] - 1s 5ms/step - loss: 56140.0156 - mae: 56140.0156 - val_loss: 51730.9688 - val_mae: 51730.9688
Epoch 4/100
250/250 [==============================] - 2s 10ms/step - loss: 51609.6797 - mae: 51609.6797 - val_loss: 50706.2656 - val_mae: 50706.2656
Epoch 5/100
250/250 [==============================] - 2s 10ms/step - loss: 51299.8828 - mae: 51299.8828 - val_loss: 50569.9336 - val_mae: 50569.9336
Epoch 6/100
250/250 [==============================] - 3s 10ms/step - loss: 51270.9531 - mae: 51270.9531 - val_loss: 50417.8672 - val_mae: 50417.8672
Epoch 7/100
250/250 [==============================] - 2s 9ms/step - loss: 51164.4922 - mae: 51164.49

In [35]:
%load_ext tensorboard

In [39]:
%tensorboard --logdir regressionlogs/fits

Reusing TensorBoard on port 6006 (pid 12308), started 0:00:13 ago. (Use '!kill 12308' to kill it.)

In [40]:
# evaluate the model on the test data
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss}, Test MAE: {test_mae}')

63/63 [==============================] - 0s 1ms/step - loss: 50300.9805 - mae: 50300.9805
Test Loss: 50300.98046875, Test MAE: 50300.98046875


In [41]:
model.save('regression_model.h5')

c:\Users\saura\OneDrive\Desktop\testing\genai\projects\ANNproject\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
